# Урок 3.2 — Базовые трансформации в Spark (DataFrame API + Spark SQL)

Этот ноутбук — шаблон урока 3.2.

В уроке работаем с полным датасетом заказов Olist:

- исходный файл: `olist_orders_dataset.csv`
- путь внутри стенда: `/data/csv/olist_orders_dataset.csv`

---

## Идея урока

Мы сделаем **одну и ту же аналитику** двумя способами:

1) через **DataFrame API**  
2) через **Spark SQL**

И параллельно закрепляем инженерный ритм:

**read → inspect → transform → validate → write → read-back**

---

## Важная привычка

Выполнять ячейку можно многократно.

- DataFrame — это просто переменная в Python.
- При повторном запуске **вы переприсваиваете** переменной новый результат.
- Это нормально: так и работает интерактивная разработка в ноутбуках.

---

## Правило путей в стенде

- исходные данные читаем из `/data/csv`
- результаты уроков пишем в `/workspace`

В этом уроке все результаты складываем сюда:

- `/workspace/lesson03_02/...`


---

## 0. Импорты и SparkSession

В стенде `spark_01` Spark-сессия обычно уже доступна как `spark`.
Если переменной нет — создадим её.


In [ ]:
# Импортируем базовые модули и Spark.
# Обратите внимание: SparkSession — это "точка входа" в Spark.
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

try:
    spark  # noqa: F821
except NameError:
    spark = SparkSession.builder.getOrCreate()

spark


---

## 1. Пути и простые хелперы

Чтобы не повторять одно и то же руками, заведём пару функций:

- `reset_dir(path)` — очистить папку перед записью (чтобы перезапуск был идемпотентным)
- `read_csv_dir(path)` и `read_parquet_dir(path)` — читать результат обратно


In [ ]:
# Настраиваем базовые пути:
# /data/csv — вход (исходные CSV)
# /workspace — выход (результаты)
DATA_DIR = "/data/csv"
WORKSPACE_DIR = "/workspace"
LESSON_DIR = f"{WORKSPACE_DIR}/lesson03_02"

os.makedirs(LESSON_DIR, exist_ok=True)

ORDERS_CSV = f"{DATA_DIR}/olist_orders_dataset.csv"

(DATA_DIR, WORKSPACE_DIR, LESSON_DIR, ORDERS_CSV)


In [ ]:
# reset_dir удаляет папку (если она есть), чтобы запись была идемпотентной.
# Идемпотентно = можно запускать много раз и получать один и тот же результат.
def reset_dir(path: str) -> None:
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)


In [ ]:
# Read-back хелперы: читаем результат обратно из папки и считаем строки.
def read_csv_dir(path: str):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

def read_parquet_dir(path: str):
    return spark.read.parquet(path)


---

## 2. Читаем данные из `/data/csv`

Читаем полный CSV с заказами.

Для обучения используем автоопределение схемы (`inferSchema=True`).
В продакшене схему обычно фиксируют явно.


In [ ]:
# Читаем CSV в DataFrame.
# df_orders — это "таблица" в Spark, но в виде DataFrame.
df_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ORDERS_CSV)
)

df_orders


---

## 3. Inspect: быстро понять, что внутри

Минимальный набор проверок, который стоит делать почти всегда:

- список колонок
- схема
- несколько строк
- количество строк


In [ ]:
# df.columns — список названий колонок.
# len(df.columns) — сколько колонок.
len(df_orders.columns), df_orders.columns


In [ ]:
# printSchema() печатает типы данных (очень полезно для понимания).
df_orders.printSchema()


In [ ]:
# show(n) — показать несколько строк.
# truncate=False — не обрезать длинные значения.
df_orders.show(5, truncate=False)


In [ ]:
# count() — действие (action). Оно запускает вычисления в Spark.
orders_cnt = df_orders.count()
orders_cnt


---

## 4. База DataFrame API: «через точку вызываем команды»

DataFrame — это объект, у которого есть методы.

Примеры:

- `df.show()` — показать строки
- `df.select(...).show()` — сначала выбрать колонки, потом показать
- `df.filter(...).select(...).show()` — несколько шагов подряд

Важно: когда пишем цепочку в несколько строк, используем:

- либо **круглые скобки** `(...)`
- либо в конце строки ставим `\`

В этом уроке будем использовать **скобки**.


In [ ]:
# 1) select — выбрать несколько колонок.
# Обратите внимание: мы НЕ меняем df_orders, а получаем новый DataFrame.
df_orders.select("order_id", "customer_id", "order_status").show(5, truncate=False)


In [ ]:
# 2) withColumn — добавить (или переопределить) колонку.
# Здесь создадим колонку order_purchase_date (дата без времени).
df_with_date = df_orders.withColumn(
    "order_purchase_date",
    F.to_date("order_purchase_timestamp")
)

df_with_date.select("order_id", "order_purchase_timestamp", "order_purchase_date").show(5, truncate=False)


In [ ]:
# 3) filter — отфильтровать строки.
# Оставим только delivered заказы.
df_delivered = df_orders.filter(F.col("order_status") == "delivered")

df_delivered.select("order_id", "order_status").show(5, truncate=False)


In [ ]:
# 4) orderBy — сортировка.
# Отсортируем delivered заказы по времени покупки по убыванию.
(
    df_delivered
    .select("order_id", "order_purchase_timestamp", "order_status")
    .orderBy(F.col("order_purchase_timestamp").desc())
    .show(5, truncate=False)
)


In [ ]:
# 5) distinct — уникальные значения.
# Посмотрим, какие статусы вообще есть в данных.
(
    df_orders
    .select("order_status")
    .distinct()
    .orderBy(F.col("order_status").asc())
    .show(20, truncate=False)
)


---

## 5. Transformations vs Actions (ленивость Spark)

В Spark многие операции **ленивые**:

- трансформации (transformations) строят план вычислений
- действия (actions) реально запускают вычисления

Посмотрим это на маленьком примере.


In [ ]:
# Создаём новый DataFrame через трансформации (select → groupBy → count).
# Пока мы НЕ вызываем action — Spark не обязан ничего считать.
df_tmp = (
    df_orders
    .select("order_status")
    .groupBy("order_status")
    .count()
)

df_tmp


In [ ]:
# explain() показывает план выполнения.
# Это НЕ действие в смысле вычисления всех данных — это "посмотреть план".
df_tmp.explain("formatted")


In [ ]:
# show() — это действие (action): Spark реально считает данные и выводит результат.
df_tmp.show(truncate=False)


---

## 6. Контракт данных: собираем удобную «рабочую» таблицу

В аналитике и инженерии часто делают понятный слой-выжимку,
с которым дальше удобно работать.

Контракт результата для заказов:

- `order_id`
- `customer_id`
- `order_status`
- `order_purchase_timestamp`
- `order_delivered_customer_date`
- `order_purchase_date` — дата покупки без времени

Сначала сделаем это через DataFrame API.


In [ ]:
# Шаг 1: выбираем нужные колонки (select).
df_orders_contract_df = df_orders.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
)

df_orders_contract_df.show(5, truncate=False)


In [ ]:
# Шаг 2: добавляем колонку с датой без времени (withColumn + to_date).
df_orders_contract_df = df_orders_contract_df.withColumn(
    "order_purchase_date",
    F.to_date("order_purchase_timestamp")
)

df_orders_contract_df.show(5, truncate=False)


In [ ]:
# Быстрый sanity-check по контракту:
# - строки не должны потеряться
# - даты должны выглядеть разумно
df_orders_contract_df.select(
    F.count("*").alias("rows_cnt"),
    F.min("order_purchase_date").alias("min_purchase_date"),
    F.max("order_purchase_date").alias("max_purchase_date"),
).show(truncate=False)


---

## 7. Несколько трансформаций подряд (один «пайплайн»)

Теперь покажем типичный стиль DataFrame API: несколько шагов подряд.

Сделаем мини-пайплайн:

1) берём контракт  
2) фильтруем только `delivered`  
3) выбираем 3 поля  
4) сортируем по дате покупки по убыванию


In [ ]:
# Собираем пайплайн в одну цепочку.
# Обратите внимание: каждая строка — понятный маленький шаг.
df_delivered_preview = (
    df_orders_contract_df
    .filter(F.col("order_status") == "delivered")
    .select("order_id", "customer_id", "order_purchase_date")
    .orderBy(F.col("order_purchase_date").desc())
)

df_delivered_preview.show(5, truncate=False)


In [ ]:
# validate: посчитаем строки — это action.
delivered_cnt = df_delivered_preview.count()
delivered_cnt


---

## 8. Агрегации: groupBy + agg

Соберём маленькую агрегированную таблицу:

- `order_status`
- `cnt`

Сделаем это через DataFrame API.


In [ ]:
# groupBy + agg — стандартный паттерн для агрегаций.
df_status_df = (
    df_orders
    .groupBy("order_status")
    .agg(F.count("*").alias("cnt"))
    .orderBy(F.col("cnt").desc(), F.col("order_status").asc())
)

df_status_df.show(truncate=False)


---

## 9. То же самое через Spark SQL

Одна важная привычка:

1) зарегистрировать DataFrame как temp view  
2) писать SQL поверх этого view

Назовём view просто `orders`.


In [ ]:
# Регистрируем DataFrame как временное представление (temp view).
# Теперь можно писать spark.sql("SELECT ... FROM orders").
df_orders.createOrReplaceTempView("orders")


In [ ]:
# SQL-версия контракта.
# to_date(...) превращает timestamp в date (без времени).
query_contract = '''
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    order_delivered_customer_date,
    to_date(order_purchase_timestamp) AS order_purchase_date
FROM orders
'''
df_orders_contract_sql = spark.sql(query_contract)

df_orders_contract_sql.show(5, truncate=False)


In [ ]:
# validate: сравним количество строк в контракте DF API и SQL.
df_orders_contract_df.count(), df_orders_contract_sql.count(), orders_cnt


In [ ]:
# SQL-версия агрегированного статуса.
query_status = '''
SELECT
    order_status,
    COUNT(*) AS cnt
FROM orders
GROUP BY order_status
ORDER BY cnt DESC, order_status ASC
'''
df_status_sql = spark.sql(query_status)

df_status_sql.show(truncate=False)


In [ ]:
# SQL-версия пайплайна "delivered preview" (как в разделе 7).
query_delivered_preview = '''
SELECT
    order_id,
    customer_id,
    to_date(order_purchase_timestamp) AS order_purchase_date
FROM orders
WHERE order_status = 'delivered'
ORDER BY order_purchase_date DESC
'''
df_delivered_preview_sql = spark.sql(query_delivered_preview)

df_delivered_preview_sql.show(5, truncate=False)


---

## 10. Запись результатов в `/workspace/lesson03_02/...`

Важно:

- Spark пишет в **директорию** и создаёт несколько `part-...` файлов  
- это нормально (и правильно) для распределённых вычислений  
- `coalesce(1)` используйте только для маленьких результатов


In [ ]:
# Куда писать контракт в CSV.
OUT_CONTRACT_CSV = f"{LESSON_DIR}/orders_contract_csv"
OUT_CONTRACT_CSV


In [ ]:
# Пишем CSV. Перед записью чистим папку.
reset_dir(OUT_CONTRACT_CSV)

(
    df_orders_contract_df
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(OUT_CONTRACT_CSV)
)

os.listdir(OUT_CONTRACT_CSV)[:5]


In [ ]:
# Куда писать контракт в Parquet.
OUT_CONTRACT_PARQUET = f"{LESSON_DIR}/orders_contract_parquet"
OUT_CONTRACT_PARQUET


In [ ]:
# Пишем Parquet. Перед записью чистим папку.
reset_dir(OUT_CONTRACT_PARQUET)

(
    df_orders_contract_df
    .write
    .mode("overwrite")
    .parquet(OUT_CONTRACT_PARQUET)
)

os.listdir(OUT_CONTRACT_PARQUET)[:5]


In [ ]:
# Куда писать агрегированные статусы (маленькая таблица).
OUT_STATUS_PARQUET = f"{LESSON_DIR}/status_cnt_parquet"
OUT_STATUS_PARQUET


In [ ]:
# Для маленьких таблиц можно сделать 1 файл (coalesce(1)).
reset_dir(OUT_STATUS_PARQUET)

(
    df_status_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .parquet(OUT_STATUS_PARQUET)
)

os.listdir(OUT_STATUS_PARQUET)


---

## 11. Read-back проверка: прочитать обратно и сравнить `count()`

Это базовая инженерная привычка:

если вы записали результат — вы должны уметь его прочитать обратно и проверить размер.


In [ ]:
# Читаем контракт обратно из CSV и сравниваем count().
df_contract_csv_back = read_csv_dir(OUT_CONTRACT_CSV)
cnt_csv_back = df_contract_csv_back.count()

cnt_csv_back, df_orders_contract_df.count()


In [ ]:
# Читаем контракт обратно из Parquet и сравниваем count().
df_contract_parquet_back = read_parquet_dir(OUT_CONTRACT_PARQUET)
cnt_parquet_back = df_contract_parquet_back.count()

cnt_parquet_back, df_orders_contract_df.count()


In [ ]:
# Маленький sanity-check: min/max timestamp должны быть похожи на исходные.
df_contract_parquet_back.select(
    F.count("*").alias("cnt"),
    F.min("order_purchase_timestamp").alias("min_ts"),
    F.max("order_purchase_timestamp").alias("max_ts"),
).show(truncate=False)


---

# Практика

Правило для всех заданий ниже:

1) чётко используйте указанный входной DataFrame  
2) сделайте трансформацию  
3) покажите `show(5, truncate=False)`  
4) запишите результат в указанную папку  
5) прочитайте результат обратно и сравните `count()`

Источник данных для заданий:

- `df_orders` — полный датасет заказов  
- `df_orders_contract_df` — контрактная таблица (если в задании указано)  
- SQL через temp view `orders` (если в задании указано)


## Задание 1. Узкая таблица заказов + переименование колонки

Входные данные:
- `df_orders`

Что сделать:
- выбрать колонки: `order_id`, `order_status`, `order_purchase_timestamp`
- переименовать `order_purchase_timestamp` в `purchase_ts`

Что вывести:
- `order_id`, `order_status`, `purchase_ts`

Куда записать:
- Parquet в `/workspace/lesson03_02/task01_orders_narrow_parquet`

Критерии приёмки:
- `show(5, truncate=False)`
- `count()` после чтения parquet совпадает


In [ ]:
TASK01_OUT = f"{LESSON_DIR}/task01_orders_narrow_parquet"

# 1) Трансформация
df_task01 = read_csv_dir(ORDERS_CSV)\
    .select("order_id", "order_status", "order_purchase_timestamp")\
    .withColumnRenamed("order_purchase_timestamp", "purchase_ts")

# 2) Проверка глазами
df_task01.show(5, truncate=False)

# 3) Запись
reset_dir(TASK01_OUT)
(df_task01.write.mode("overwrite").parquet(TASK01_OUT))

# 4) Read-back + count()
back = read_parquet_dir(TASK01_OUT)
print(f"Read before: {df_task01.count()}, read after: {back.count()}")

## Задание 2. Флаг доставки

Входные данные:
- `df_orders`

Что сделать:
- добавить колонку `is_delivered`:
  - 1 если `order_status = 'delivered'`
  - 0 иначе

Что вывести:
- все колонки контракта + `is_delivered`

Куда записать:
- CSV (с заголовком) в `/workspace/lesson03_02/task02_orders_with_flag_csv`

Критерии приёмки:
- `show(5, truncate=False)`
- `is_delivered` принимает только 0/1
- `count()` после чтения csv совпадает


In [ ]:
TASK02_OUT = f"{LESSON_DIR}/task02_orders_with_flag_csv"

# 1) Трансформация
df_task02 = read_csv_dir(ORDERS_CSV)
df_task02 = (
    df_task02
    .withColumn("delivered_flag",
    F.when(F.col("order_status") == 'delivered', 1).otherwise(0))
    .select("order_id", "customer_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "delivered_flag")
)
# 2) show
df_task02.show(5, truncate=False)

# 3) Запись CSV
reset_dir(TASK02_OUT)
(df_task02.write.mode("overwrite").option("header", True).csv(TASK02_OUT))

# 4) Read-back + count()
back = read_csv_dir(TASK02_OUT)
print(f"Read before: {df_task02.count()}, read after: {back.count()}")

## Задание 3. Фильтр по статусам

Входные данные:
- `df_orders`

Что сделать:
- оставить только статусы: `delivered`, `shipped`, `processing`

Что вывести:
- `order_id`, `order_status`, `order_purchase_timestamp`

Куда записать:
- Parquet в `/workspace/lesson03_02/task03_active_statuses_parquet`

Критерии приёмки:
- в результате нет других статусов
- `count()` после чтения parquet совпадает


In [ ]:
TASK03_OUT = f"{LESSON_DIR}/task03_active_statuses_parquet"

df_task03 = read_csv_dir(ORDERS_CSV)

df_select = df_task03.filter(F.col("order_status").isin(["delivered", "shipped", "processing"])).select("order_id","order_status","order_purchase_timestamp")

df_select.show(5, truncate=False)

reset_dir(TASK03_OUT)

(
    df_select
    .write
    .mode("overwrite")
    .parquet(TASK03_OUT)
)

back = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .parquet(TASK03_OUT)
)
print(f"Read before: {df_select.count()}, read after: {back.count()}")


## Задание 4. Фильтр по дате покупки

Входные данные:
- `df_orders`

Что сделать:
- оставить заказы, где `order_purchase_date >= '2017-01-01'`

Что вывести:
- `order_id`, `customer_id`, `order_purchase_date`, `order_status`

Куда записать:
- CSV (с заголовком) в `/workspace/lesson03_02/task04_recent_orders_csv`

Критерии приёмки:
- минимальная дата в результате не раньше 2017-01-01
- `count()` после чтения csv совпадает


In [ ]:
TASK04_OUT = f"{LESSON_DIR}/task04_recent_orders_csv"

df_task04 = read_csv_dir(ORDERS_CSV)

df_with_date = df_orders.withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))

df_select = df_with_date.filter(F.col("order_purchase_date") >= F.lit("2017-01-01")).select("order_id", "customer_id", "order_purchase_date", "order_status")
df_select.show(5, truncate=False)

reset_dir(TASK04_OUT)
(df_select.write.mode("overwrite").option("header", True).csv(TASK04_OUT))

back = read_csv_dir(TASK04_OUT)
print(f"Read before: {df_select.count()}, read after: {back.count()}")

## Задание 5. Первая покупка клиента

Входные данные:
- `df_orders`

Что сделать:
- для каждого `customer_id` найти минимальный `order_purchase_timestamp`

Что вывести:
- `customer_id`
- `first_purchase_ts` — минимальный `order_purchase_timestamp`

Сортировка:
- по `first_purchase_ts` по возрастанию

Куда записать:
- Parquet в `/workspace/lesson03_02/task05_first_purchase_parquet`

Критерии приёмки:
- `customer_id` уникален
- `count()` после чтения parquet совпадает


In [ ]:
TASK05_OUT = f"{LESSON_DIR}/task05_first_purchase_parquet"

df_task05 = read_csv_dir(ORDERS_CSV)
df_select = df_task05.groupBy("customer_id").agg(F.min("order_purchase_timestamp").alias("first_purchase_ts")).orderBy("first_purchase_ts")
df_select.show(5, truncate=False)

(
    df_select
    .write
    .mode("overwrite")
    .parquet(TASK05_OUT)
)

back = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .parquet(TASK05_OUT)
)

print(f"Read before: {df_select.count()}, read after: {back.count()}")

## Задание 6. Количество заказов по дням

Входные данные:
- `df_orders_contract_df`

Что сделать:
- сгруппировать по `order_purchase_date`
- посчитать количество заказов `orders_cnt`

Что вывести:
- `order_purchase_date`
- `orders_cnt`

Сортировка:
- по `order_purchase_date` по возрастанию

Куда записать:
- Parquet в `/workspace/lesson03_02/task06_daily_orders_parquet`

Критерии приёмки:
- `count()` строк равно количеству уникальных дат
- `count()` после чтения parquet совпадает


In [ ]:
TASK06_OUT = f"{LESSON_DIR}/task06_daily_orders_parquet"

df_task06 = read_csv_dir(ORDERS_CSV)
df_with_date = df_task06\
    .withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))\
    .groupBy("order_purchase_date")\
    .agg(F.count("order_id").alias("orders_cnt"))\
    .orderBy("order_purchase_date")

df_with_date.show(5, truncate=False)

(
    df_with_date
    .write
    .mode("overwrite")
    .parquet(TASK06_OUT)
)

back = (
    spark.read
    .option("header", True)
    .option("infernalSchema", True)
    .parquet(TASK06_OUT)
)

print(f"Read before: {df_with_date.count()}, read after: {back.count()}")

## Задание 7. Визуальная выборка для проверки

Входные данные:
- `df_orders_contract_df`

Что сделать:
- выбрать колонки: `order_id`, `order_status`, `order_purchase_date`
- отсортировать:
  - `order_purchase_date` по убыванию
  - внутри даты `order_status` по возрастанию

Что вывести:
- `order_id`, `order_status`, `order_purchase_date`

Куда записать:
- CSV (с заголовком) в `/workspace/lesson03_02/task07_sorted_preview_csv`

Критерии приёмки:
- первая строка имеет максимальную дату
- `count()` после чтения csv совпадает


In [ ]:
TASK07_OUT = f"{LESSON_DIR}/task07_sorted_preview_csv"

df_task07 = read_csv_dir(ORDERS_CSV)\
    .withColumn("order_purchase_date", F.to_date("order_purchase_timestamp"))\
    .select("order_id", "order_status", "order_purchase_date")\
    .orderBy(F.desc("order_purchase_date"), "order_status")

df_with_date.show(5, truncate=False)

reset_dir(TASK07_OUT)
(df_with_date.write.mode("overwrite").option("header", True).csv(TASK07_OUT))

back = read_csv_dir(TASK07_OUT)
print(f"Read before: {df_with_date.count()}, read after: {back.count()}")


---

# ✅ Проверки

Запускайте check-ячейки ниже: получите **OK** или **НЕ OK** + причину.


In [ ]:
# Импортируем утилиты для проверок и определяем пути для результатов заданий.
import os
from pyspark.sql import functions as F

# ---------- Пути заданий ----------
TASK01_OUT = f"{LESSON_DIR}/task01_orders_narrow_parquet"
TASK02_OUT = f"{LESSON_DIR}/task02_orders_with_flag_csv"
TASK03_OUT = f"{LESSON_DIR}/task03_active_statuses_parquet"
TASK04_OUT = f"{LESSON_DIR}/task04_recent_orders_csv"
TASK05_OUT = f"{LESSON_DIR}/task05_first_purchase_parquet"
TASK06_OUT = f"{LESSON_DIR}/task06_daily_orders_parquet"
TASK07_OUT = f"{LESSON_DIR}/task07_sorted_preview_csv"

# ---------- Утилиты ----------
def ok(task: str, details: str = "") -> None:
    msg = f"OK — {task}"
    if details:
        msg += f" | {details}"
    print(msg)

def bad(task: str, err: Exception) -> None:
    print(f"НЕ OK — {task} | {err}")

def run_check(task: str, check_fn) -> None:
    try:
        check_fn()
        ok(task)
    except AssertionError as e:
        bad(task, e)

def assert_exists(path: str) -> None:
    assert os.path.exists(path), f"путь не найден: {path}"


## Проверка задания 1 — узкая таблица + rename (Parquet)


In [ ]:
# Проверяем: есть ли папка, правильные колонки, правильный count и совпадают min/max timestamps.
def check_task01():
    assert_exists(TASK01_OUT)
    back = read_parquet_dir(TASK01_OUT)

    assert back.columns == ["order_id", "order_status", "purchase_ts"], "ожидаются колонки: order_id, order_status, purchase_ts"
    assert back.count() == df_orders.count(), "count() должен совпадать с df_orders"

    src = df_orders.agg(
        F.min("order_purchase_timestamp").alias("mn"),
        F.max("order_purchase_timestamp").alias("mx"),
    ).first()
    tgt = back.agg(
        F.min("purchase_ts").alias("mn"),
        F.max("purchase_ts").alias("mx"),
    ).first()

    assert str(src["mn"]) == str(tgt["mn"]), "min timestamp не совпал"
    assert str(src["mx"]) == str(tgt["mx"]), "max timestamp не совпал"

run_check("TASK01", check_task01)


## Проверка задания 2 — флаг доставки (CSV)


In [ ]:
# Проверяем: есть колонка is_delivered, count совпадает, значения только 0/1, delivered согласуется.
def check_task02():
    assert_exists(TASK02_OUT)
    back = read_csv_dir(TASK02_OUT)

    assert "is_delivered" in back.columns, "нет колонки is_delivered"
    assert back.count() == df_orders_contract_df.count(), "count() должен совпадать с df_orders_contract_df"

    bad_vals = back.filter(~F.col("is_delivered").isin([0, 1])).count()
    assert bad_vals == 0, "is_delivered должен принимать только 0/1"

    expected_delivered = df_orders_contract_df.filter(F.col("order_status") == "delivered").count()
    got_delivered = back.filter(F.col("is_delivered") == 1).count()
    assert got_delivered == expected_delivered, "кол-во is_delivered=1 не совпало с delivered"

run_check("TASK02", check_task02)


## Проверка задания 3 — фильтр по статусам (Parquet)


In [ ]:
# Проверяем: колонки, статусы только из списка, и count совпадает с ожидаемым.
def check_task03():
    assert_exists(TASK03_OUT)
    back = read_parquet_dir(TASK03_OUT)

    expected_statuses = ["delivered", "shipped", "processing"]

    assert back.columns == ["order_id", "order_status", "order_purchase_timestamp"], "неверные колонки/порядок"
    bad_status_cnt = back.filter(~F.col("order_status").isin(expected_statuses)).count()
    assert bad_status_cnt == 0, "в результате есть статусы кроме delivered/shipped/processing"

    expected_cnt = df_orders.filter(F.col("order_status").isin(expected_statuses)).count()
    assert back.count() == expected_cnt, "count() не совпал с ожидаемым после фильтра"

run_check("TASK03", check_task03)


## Проверка задания 4 — фильтр по дате покупки (CSV)


In [ ]:
# Проверяем: колонки, минимальная дата не раньше 2017-01-01, count совпадает с ожидаемым.
def check_task04():
    assert_exists(TASK04_OUT)
    back = read_csv_dir(TASK04_OUT)

    assert back.columns == ["order_id", "customer_id", "order_purchase_date", "order_status"], "неверные колонки/порядок"

    min_dt = back.select(F.min("order_purchase_date").alias("mn")).first()["mn"]
    assert str(min_dt) >= "2017-01-01", f"минимальная дата раньше 2017-01-01: {min_dt}"

    expected_cnt = df_orders_contract_df.filter(F.col("order_purchase_date") >= F.lit("2017-01-01")).count()
    assert back.count() == expected_cnt, "count() не совпал с ожидаемым после фильтра даты"

run_check("TASK04", check_task04)


## Проверка задания 5 — первая покупка клиента (Parquet)


In [ ]:
# Проверяем: колонки, уникальность customer_id, count совпадает и значения first_purchase_ts совпадают.
def check_task05():
    assert_exists(TASK05_OUT)
    back = read_parquet_dir(TASK05_OUT)

    assert back.columns == ["customer_id", "first_purchase_ts"], "ожидаются колонки: customer_id, first_purchase_ts"
    assert back.count() == back.select("customer_id").distinct().count(), "customer_id должен быть уникален"

    expected = (
        df_orders.groupBy("customer_id")
        .agg(F.min("order_purchase_timestamp").alias("first_purchase_ts"))
    )

    assert back.count() == expected.count(), "count() не совпал с ожидаемым числом клиентов"

    exp_s = expected.select(
        "customer_id",
        F.col("first_purchase_ts").cast("string").alias("first_purchase_ts")
    )
    got_s = back.select(
        "customer_id",
        F.col("first_purchase_ts").cast("string").alias("first_purchase_ts")
    )

    mismatch = (
        exp_s.alias("e")
        .join(got_s.alias("g"), on="customer_id", how="left")
        .filter(
            (F.col("g.first_purchase_ts").isNull())
            | (F.col("e.first_purchase_ts") != F.col("g.first_purchase_ts"))
        )
        .count()
    )
    assert mismatch == 0, f"есть расхождения в first_purchase_ts (проблемных customer_id: {mismatch})"

run_check("TASK05", check_task05)


## Проверка задания 6 — количество заказов по дням (Parquet)


In [ ]:
# Проверяем: колонки, число строк равно числу уникальных дат, и orders_cnt совпадают по датам.
def check_task06():
    assert_exists(TASK06_OUT)
    back = read_parquet_dir(TASK06_OUT)

    assert back.columns == ["order_purchase_date", "orders_cnt"], "ожидаются колонки: order_purchase_date, orders_cnt"
    assert back.count() == df_orders_contract_df.select("order_purchase_date").distinct().count(), "число строк должно равняться числу уникальных дат"

    expected = (
        df_orders_contract_df.groupBy("order_purchase_date")
        .agg(F.count("*").alias("orders_cnt"))
    )

    exp_s = expected.select(
        F.col("order_purchase_date").cast("string").alias("order_purchase_date"),
        F.col("orders_cnt").cast("long").alias("orders_cnt"),
    )
    got_s = back.select(
        F.col("order_purchase_date").cast("string").alias("order_purchase_date"),
        F.col("orders_cnt").cast("long").alias("orders_cnt"),
    )

    mismatch = (
        exp_s.alias("e")
        .join(got_s.alias("g"), on="order_purchase_date", how="left")
        .filter((F.col("g.orders_cnt").isNull()) | (F.col("e.orders_cnt") != F.col("g.orders_cnt")))
        .count()
    )
    assert mismatch == 0, f"есть расхождения в orders_cnt по датам (проблемных дат: {mismatch})"

run_check("TASK06", check_task06)


## Проверка задания 7 — preview сортировка (CSV)


In [ ]:
# Проверяем: колонки, count совпадает.
# Порядок строк после write/read не гарантирован, поэтому проверяем "первую строку" только после явной сортировки.
def check_task07():
    assert_exists(TASK07_OUT)
    back = read_csv_dir(TASK07_OUT)

    assert back.columns == ["order_id", "order_status", "order_purchase_date"], "неверные колонки/порядок"
    assert back.count() == df_orders_contract_df.count(), "count() должен совпадать с df_orders_contract_df"

    expected_first = (
        df_orders_contract_df
        .select("order_id", "order_status", "order_purchase_date")
        .orderBy(F.col("order_purchase_date").desc(), F.col("order_status").asc())
        .first()
    )
    back_first = (
        back
        .select("order_id", "order_status", "order_purchase_date")
        .orderBy(F.col("order_purchase_date").desc(), F.col("order_status").asc())
        .first()
    )

    assert str(back_first["order_purchase_date"]) == str(expected_first["order_purchase_date"]), "первая строка должна иметь максимальную дату"
    assert str(back_first["order_status"]) == str(expected_first["order_status"]), "внутри максимальной даты порядок по status должен совпадать"
    assert str(back_first["order_id"]) == str(expected_first["order_id"]), "order_id первой строки не совпал с ожидаемым"

run_check("TASK07", check_task07)
